In [1]:
!pip install fastapi uvicorn python-multipart scikit-learn joblib

## Task 1.1: Load and Prepare Data

In [2]:
import os
import pytesseract
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

def load_documents(data_dir):
    """
    Load documents from folder structure and extract text via OCR.
    """
    documents = []
    labels = []
    
    # Check if the directory exists to avoid errors
    if not os.path.exists(data_dir):
        print(f"Error: The directory {data_dir} does not exist.")
        return documents, labels

    for doc_type in os.listdir(data_dir):
        folder_path = os.path.join(data_dir, doc_type)
        
        # Only process directories (invoices, receipts, etc.)
        if not os.path.isdir(folder_path):
            continue
            
        for filename in os.listdir(folder_path):
            # Process common image formats
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
                file_path = os.path.join(folder_path, filename)
                
                try:
                    # Extract text via OCR
                    img = Image.open(file_path).convert('RGB')
                    text = pytesseract.image_to_string(img)
                    
                    if text.strip():  # Only add if OCR found text
                        documents.append(text)
                        labels.append(doc_type)
                except Exception as e:
                    print(f"Could not process {file_path}: {e}")
                    
    return documents, labels

# Load data - ensure the folder 'training_data' exists and has subfolders
documents, labels = load_documents('/kaggle/input/datasets/ayeshaameer/ocr-complete-dataset/OCR_Complete_dataset/train')

if documents:
    print(f'Loaded {len(documents)} documents')
    print(f'Classes: {set(labels)}')
else:
    print("No documents were loaded. Check your folder structure.")

Loaded 656 documents
Classes: {'receipts', 'form', 'invoice', 'resume', 'memo'}


## Task 1.2: Train Classifier

In [3]:
# 1. Split data
# Added a check to ensure you have enough data for stratification
try:
    X_train, X_test, y_train, y_test = train_test_split(
        documents, 
        labels,
        test_size=0.2,
        random_state=42,
        stratify=labels  # Ensures equal distribution of classes
    )
except ValueError as e:
    print(f"Stratify Error: {e}")
    print("Falling back to non-stratified split...")
    X_train, X_test, y_train, y_test = train_test_split(
        documents, 
        labels,
        test_size=0.2,
        random_state=42
    )

# 2. Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2)  # Captures phrases like "Invoice Number"
)

# 3. Fit and transform
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 4. Train classifier
classifier = LogisticRegression(max_iter=1000, multi_class='auto')
classifier.fit(X_train_vec, y_train)

# 5. Evaluate
y_pred = classifier.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.2%}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))


Accuracy: 83.33%

Classification Report:
              precision    recall  f1-score   support

        form       0.74      0.78      0.76        32
     invoice       0.72      0.88      0.79        32
        memo       0.96      0.81      0.88        32
    receipts       0.00      0.00      0.00         4
      resume       0.97      0.97      0.97        32

    accuracy                           0.83       132
   macro avg       0.68      0.69      0.68       132
weighted avg       0.82      0.83      0.82       132



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: 

## Task 1.3: Save Model

In [4]:
# ================= SAVE TRAINED MODELS =================

import os
import joblib


# Save trained vectorizer and classifier
joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(classifier, "classifier.pkl")

print("\nModels saved as 'vectorizer.pkl' and 'classifier.pkl'")



Models saved as 'vectorizer.pkl' and 'classifier.pkl'


# PART 2: BUILD REST API

## Task 2.1 to 2.4 is done in app.py file 

#  Project Documentation: Document Intelligence & OCR System

## 1. Project Overview
This project presents a complete pipeline for **OCR-based receipt text extraction** combined with **image preprocessing** and **deep learning-based digit recognition**. The system has been evolved from a research notebook into a **production-ready FastAPI application**, allowing for real-time document classification and information extraction.



##  Deployment & API Reference

###  Execution Commands
To run this system in **GitHub Codespaces**, the following terminal commands were used:

**System Requirements:**
```bash
sudo apt-get update && sudo apt-get install tesseract-ocr -y
python -m spacy download en_core_web_sm
pip install fastapi uvicorn python-multipart pytesseract pillow scikit-learn joblib spacy

## Starting the Server:
### Force clear port 8000 if already in use
**fuser -k 8000/tcp** 

### Launch the FastAPI app
**uvicorn app:app --host 0.0.0.0 --port 8000 --reload**

## API Access & Endpoints
 **Base URL: https://roseanne-harsh-jacob.ngrok-free.dev**

 **Interactive UI: Append /docs to the URL to access Swagger UI.**